# 表单提交与原生校验

学习目标：能核对表单实际发送的数据，选择提交方式和原生约束，并说明浏览器校验与服务端校验的边界。

前置知识：表单、控件标签、name/value、按钮类型，以及本机 HTTP 页面和浏览器开发者工具的基本操作。

适用范围：WHATWG HTML Living Standard；使用现代浏览器，pattern 示例按 v 标志规则编写。浏览器提示文案、自动填充和虚拟键盘可能因系统与设置不同而变化。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/11-form-submission-and-validation/。

1. [submission.html](scripts/11-form-submission-and-validation/submission.html)：GET、POST、提交按钮与字段取舍。
2. [upload.html](scripts/11-form-submission-and-validation/upload.html)、[sample-note.txt](scripts/11-form-submission-and-validation/sample-note.txt)：文件上传与原创模拟素材。
3. [constraints.html](scripts/11-form-submission-and-validation/constraints.html)：必填、范围、步长和长度。
4. [pattern.html](scripts/11-form-submission-and-validation/pattern.html)：格式约束、自动填充和键盘提示。
5. [echo_server.py](scripts/11-form-submission-and-validation/echo_server.py)：本机页面服务与请求回显，保留重复字段和空值；不保存上传文件、不解析 multipart、不做业务校验。

## 打开配套页面

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/html
```

Step 2：启动本章预览服务。

```bash
python scripts/11-form-submission-and-validation/echo_server.py
```

Step 3：打开[本章提交页](http://127.0.0.1:8011/submission.html)。

服务固定绑定 127.0.0.1:8011，只把本章指定文件映射到站点根路径；它不是以 content/Web与应用开发/html 为根的通用静态服务。

Notebook 配套文件链接相对于本 Notebook；HTML 中的 echo 相对于当前页面地址，例如从 /submission.html 解析为 /echo。该地址由服务器处理，不是磁盘上的文件。

只使用模拟数据。接收端仅回显，单次请求体最多 64 KiB（65,536 字节）；上传使用本章的小文本。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 提交地址与方法

### 1.1 GET 与查询参数

提交表单时，浏览器根据控件的 name 和当前值构造数据，再发送到指定地址。

- action：接收地址，本例的 echo 是本机回显接口。
- method：提交方法。get 把数据放入 URL 查询部分。

```html
<form action="echo" method="get">
  <label for="basic-query">查询词</label>
  <input id="basic-query" name="q" value="HTML">
  <button type="submit">GET 基本提交</button>
</form>
<!-- 保持初值提交：地址为 /echo?q=HTML，回显方法为 GET，请求体为空。 -->
```

q 是本例的字段名，HTML 是输入框的初始值。地址中的问号开始查询部分，q=HTML 表示这对名称和值。

配套文件：[scripts/11-form-submission-and-validation/submission.html](scripts/11-form-submission-and-validation/submission.html) · [浏览器预览](http://127.0.0.1:8011/submission.html)


### 1.2 POST 与请求体编码

POST 表单把字段放在请求体中。URL 仍可以带自己的查询参数；POST 也不等于加密，传输加密依赖 HTTPS。GET 字段会出现在地址栏等位置，不适合敏感信息。

完整示例中的 POST 按钮覆盖所在表单的提交设置：

- formaction：覆盖接收地址，这里为 echo-post。
- formmethod：覆盖提交方法，这里为 post。
- formenctype：覆盖本次 POST 请求体编码。

intent 是本例自定字段名，get、post 是两个按钮的值。

```html
<button type="submit" name="intent" value="post"
        formaction="echo-post" formmethod="post"
        formenctype="application/x-www-form-urlencoded">POST 回显</button>
```

配套文件：[scripts/11-form-submission-and-validation/submission.html](scripts/11-form-submission-and-validation/submission.html) · [浏览器预览](http://127.0.0.1:8011/submission.html)

&lt;form&gt; 用 enctype 设置 POST 编码，按钮可用 formenctype 覆盖：

- application/x-www-form-urlencoded：默认编码，用名称和值表示普通字段。
- multipart/form-data：多部分表单数据，发送文件内容时使用，见第 3 节。
- text/plain：纯文本形式，存在歧义，不适合作为通用数据接口。

给 GET 表单写 multipart/form-data 不会使它上传文件内容；通过 HTTP 提交 GET 表单仍生成 URL 编码查询。

## 2 哪些控件进入请求

### 2.1 名称、值与控件状态

浏览器构造的是允许重名、有顺序的数据条目，不是把所有输入自动变成一个字典。

- 普通输入缺少 name，或 name 为空：跳过，id 不能补上。
- disabled 控件：跳过。
- readonly 控件：只读本身不排除提交，本例 ticket 仍发送。
- 可提交的具名文本框：即使内容为空，也可以产生空值条目。

以下片段都位于提交页的同一表单内。

```html
<p>
  <label for="ticket">只读编号</label>
  <input id="ticket" name="ticket" value="D11" readonly>
</p>
<p>
  <label for="memo">备注（可留空）</label>
  <input id="memo" name="memo">
</p>
<p>
  <label for="unavailable">尚未开放的字段</label>
  <input id="unavailable" name="locked" value="不可选" disabled>
</p>
<p>
  <label for="local-note">有 id、没有 name 的字段</label>
  <input id="local-note" value="仅显示">
</p>
```

配套文件：[scripts/11-form-submission-and-validation/submission.html](scripts/11-form-submission-and-validation/submission.html) · [浏览器预览](http://127.0.0.1:8011/submission.html)

选择类控件还有各自的取舍规则：

- checkbox、radio：只发送当前选中的项，未选中不是发送 false。
- checkbox 未写 value：选中时默认值为 on。
- &lt;select&gt;：收集选中且未禁用的选项，多选可以形成多个同名条目。
- &lt;output&gt;：其内容不会作为提交值发送。

本例两个选中的 topic 会分别形成条目。

```html
<fieldset>
  <legend>关注主题</legend>
  <label><input type="checkbox" name="topic" value="html" checked>HTML</label>
  <label><input type="checkbox" name="topic" value="css" checked>CSS</label>
  <label><input type="checkbox" name="topic" value="js">JavaScript</label>
</fieldset>
<!-- 初值提交：q、ticket、memo、两个 topic 和本次按钮的 intent 应进入数据。 -->
<!-- locked、无 name 的 local-note、未选中的 js 不应出现；memo 应保留空值。 -->
```

配套文件：[scripts/11-form-submission-and-validation/submission.html](scripts/11-form-submission-and-validation/submission.html) · [浏览器预览](http://127.0.0.1:8011/submission.html)

### 2.2 本次提交按钮与隐式提交

触发本次提交的元素称为提交者（submitter）。有名称的提交按钮中，只有本次提交者的 name/value 进入数据；普通按钮、重置按钮和未使用的提交按钮不进入数据。

按 Enter 可能触发隐式提交（implicit submission），常通过表单的默认提交按钮执行。本例的默认提交按钮是第一个 GET 按钮，不能把按 Enter 简单理解为“没有提交者”。比较两个请求时应明确激活对应按钮。

```html
<button type="submit" name="intent" value="get">GET 查询回显</button>
<button type="submit" name="intent" value="post"
        formaction="echo-post" formmethod="post"
        formenctype="application/x-www-form-urlencoded">POST 回显</button>
<button type="reset">恢复示例初值</button>
```

配套文件：[scripts/11-form-submission-and-validation/submission.html](scripts/11-form-submission-and-validation/submission.html) · [浏览器预览](http://127.0.0.1:8011/submission.html)

### 2.3 从 Network 核对位置与编码

在提交页按 F12 打开 Network（网络），勾选 Preserve log（保留日志），分别发送 GET 与 POST。选中 echo 或 echo-post 请求：

- Headers（标头）：查看 Request URL、Request Method、Content-Type。
- Payload（载荷）：查看查询参数或表单数据；需要原始编码时选择 view source。
- 回显页：Query entries 显示 URL 查询条目；URL-encoded body entries 显示 URL 编码请求体条目。

URL 编码中的空格通常写成 +，字面加号写成 %2B；中文按 UTF-8 字节进行百分号编码。编码不是加密，也不同于 HTML 属性中表示 & 的字符引用 &amp;amp;。

原始请求体预览与解码条目是不同视角；同名字段和空值都应保留。先恢复示例初值，再比较另一种方法。

配套文件：[scripts/11-form-submission-and-validation/submission.html](scripts/11-form-submission-and-validation/submission.html) · [浏览器预览](http://127.0.0.1:8011/submission.html)

## 3 发送文件内容

### 3.1 POST 与 multipart/form-data

原生表单要发送文件内容，使用 POST 配合 multipart/form-data。URL 编码会把文件条目转换成文件名，不能发送其内容。

文件控件放在指定这两个属性的表单中。页面里的普通“文件说明”字段也会一起发送；以下片段只摘出上传所需部分。

文件控件的属性：

- accept=".txt,text/plain"：提示文件选择器优先显示文本文件，不检验真实内容。
- required：必须选择文件。
- multiple：允许多个文件；本例未使用，只选一个。

实际接收系统仍须检查大小、类型和内容。

```html
<form action="echo" method="post" enctype="multipart/form-data">
  <p>
<label for="attachment">模拟文本文件（必选）</label>
<input id="attachment" type="file" name="attachment"
       accept=".txt,text/plain" required>
  </p>
<button type="submit" name="intent" value="upload">上传并回显</button>
</form>
```

配套文件：[scripts/11-form-submission-and-validation/upload.html](scripts/11-form-submission-and-validation/upload.html) · [浏览器预览](http://127.0.0.1:8011/upload.html)

### 3.2 查看 multipart 原始体

选择 [sample-note.txt](scripts/11-form-submission-and-validation/sample-note.txt)，点击“上传并回显”。multipart 将普通字段和文件分别放进不同部分，用浏览器生成的 boundary（边界字符串）分隔。

- Content-Type：包含 multipart/form-data 及本次 boundary。
- Content-Disposition：标明字段名，文件部分还包含文件名。
- 原始体：可以找到素材中的两行文本；整个请求体大小不等于单个文件大小。

本例服务器只提供 UTF-8 预览，不解析 multipart，也不保存上传文件。二进制文件不一定有可读的文本表示。

```html
<button type="submit" name="intent" value="upload">上传并回显</button>
<!-- 选中 sample-note.txt 后：Content-Type 应有 multipart/form-data 和 boundary。 -->
<!-- 在原始体中找 caption、attachment、filename 及文件的两行文本。 -->
<!-- intent=upload 是按钮字段；文件中的 strong 标签应原样显示，不能变成加粗内容。 -->
<!-- 检查“Body bytes received”表示整个请求体大小，不能当作单个文件大小。 -->
```

配套文件：[scripts/11-form-submission-and-validation/upload.html](scripts/11-form-submission-and-validation/upload.html) · [浏览器预览](http://127.0.0.1:8011/upload.html)

## 4 必填、范围、步长与长度

原生约束校验（constraint validation）由浏览器根据控件类型和属性检查输入。正常提交时，参与校验的字段若不满足约束，浏览器会阻止提交并提示修改。

### 4.1 必填与长度

- required：文本不能为空，文件必须选择；对 checkbox 则要求该项选中。
- minlength：文本最短长度，但不会自动要求可选字段非空。
- maxlength：文本最长长度，浏览器通常在输入时就限制继续增加内容。

以下别名必须填写 3～8 个 UTF-16 码元（code unit）。长度不一定等于肉眼看到的字符数，有些字符占两个码元。

```html
<label for="alias">虚构别名（必填，3～8 个 UTF-16 码元）</label>
<input id="alias" name="alias" type="text"
       required minlength="3" maxlength="8" autocomplete="off">
```

配套文件：[scripts/11-form-submission-and-validation/constraints.html](scripts/11-form-submission-and-validation/constraints.html) · [浏览器预览](http://127.0.0.1:8011/constraints.html)

配套页的备注不带 required，留空仍可提交，填写后才检查最短长度。测试时实际键入 ab、abc；由脚本赋值不会像用户编辑那样触发 minlength/maxlength 检查。

```html
<label for="note">可选备注（填写时 3～8 个 UTF-16 码元）</label>
<textarea id="note" name="note" minlength="3" maxlength="8"></textarea>
```

配套文件：[scripts/11-form-submission-and-validation/constraints.html](scripts/11-form-submission-and-validation/constraints.html) · [浏览器预览](http://127.0.0.1:8011/constraints.html)

### 4.2 数量范围与步长

- min：下界。
- max：上界。
- step：相对步长基准的合法间隔，不只是按钮每次增加多少。

本例 min=1、max=5、step=2，只允许 1、3、5。2 虽在范围内，仍不符合步长。

```html
<label for="copies">模拟份数（必填，只允许 1、3、5）</label>
<input id="copies" name="copies" type="number"
       min="1" max="5" step="2" required value="1">
<!-- 输入 abc、勾选确认、备注留空，份数依次改为 0、2、6：正常提交都应失败。 -->
<!-- 把份数改为 3 后应能提交；别名 abc、空 note、copies=3、confirmed=yes 应回显。 -->
```

配套文件：[scripts/11-form-submission-and-validation/constraints.html](scripts/11-form-submission-and-validation/constraints.html) · [浏览器预览](http://127.0.0.1:8011/constraints.html)

number 的步长基准优先取有效 min，其次取有效 value 内容属性，否则取 0；默认 step 是 1。step="any" 取消步长限制，但不取消上下界。其他类型的单位与基准需按该类型核对。

### 4.3 约束只对适用控件生效

required 对同组 radio 要求至少选中一项，对 checkbox 则只要求带此属性的那一项，不自动表达“本组任选一项”。本页的确认框需要单独勾选。

```html
<label>
  <input id="confirmed" name="confirmed" type="checkbox"
         value="yes" required>我确认使用模拟数据（必选）
</label>
```

配套文件：[scripts/11-form-submission-and-validation/constraints.html](scripts/11-form-submission-and-validation/constraints.html) · [浏览器预览](http://127.0.0.1:8011/constraints.html)

disabled 控件、readonly 文本等不参与原生约束校验；required 不适用于 hidden 等类型。不能向所有输入机械添加同一套约束。

隔离观察一条约束时，先把其他字段填合法。提示文字和弹窗外观可能因浏览器、语言、系统不同而变化，应核对是否阻止提交及实际请求。

## 5 pattern：完整格式约束

pattern 适用于 text、search、url、tel、email 和 password，不适用于 number 或 &lt;textarea&gt;。它按正则表达式匹配整个非空值，不是查找子串；要拒绝空值仍需 required。

本例用 [A-Z0-9&#92;-]{3,8} 限定虚构练习编号：

- A-Z：ASCII 大写字母范围。
- 0-9：ASCII 数字范围。
- &#92;-：字符类中的普通连字符。
- {3,8}：前面的允许字符重复 3～8 次。

HTML 属性中实际只写一个反斜杠，不加正则表达式的斜杠分隔符。

```html
<label for="code">练习编号（必填，3～8 位大写字母、数字或连字符）</label>
<input id="code" name="code" type="text"
       pattern="[A-Z0-9\-]{3,8}" required value="AB-12"
       autocomplete="off">
<!-- AB-12 应成功；ab-12、AB/12、AB 分别有小写、斜杠、长度问题，应失败。 -->
<!-- 清空后应因 required 失败；检查原始属性中只写一个反斜杠。 -->
```

配套文件：[scripts/11-form-submission-and-validation/pattern.html](scripts/11-form-submission-and-validation/pattern.html) · [浏览器预览](http://127.0.0.1:8011/pattern.html)

现代 HTML 按 JavaScript 正则表达式的 v 标志编译 pattern，字符类内的字面连字符等标点有转义要求；不能直接照搬旧 u 规则的示例。格式无效时，该 pattern 不形成有效约束，而不是拒绝所有输入。

要求写在可见标签里，不能只放在悬停提示中。本例规则只用于练习编号，不适合作为真实人名规则。

## 6 自动填充与键盘提示

### 6.1 inputmode 不负责校验

inputmode="numeric" 提示虚拟数字键盘，不阻止粘贴字母。六位编号不用于数值运算，使用 type="text" 保留前导零，再用 pattern 约束格式。

```html
<label for="digits">模拟数字编号（必填，六位数字）</label>
<input id="digits" name="digits" type="text" inputmode="numeric"
       pattern="[0-9]{6}" required value="001234"
       autocomplete="off">
<!-- 001234 应原样回显；粘贴 00a234 后，应由 pattern 阻止提交。 -->
<!-- 在移动设备观察键盘；桌面窗口变窄不能证明真实触屏键盘符合预期。 -->
```

配套文件：[scripts/11-form-submission-and-validation/pattern.html](scripts/11-form-submission-and-validation/pattern.html) · [浏览器预览](http://127.0.0.1:8011/pattern.html)

### 6.2 autocomplete 描述字段用途

- autocomplete="nickname"：提示这是昵称字段。
- autocomplete="off"：请求不自动填充，但密码管理器等可能仍为登录字段提供填充。

是否出现候选值还取决于浏览器、设置与已有资料。autocomplete 不增加必填或格式约束，也不是保密机制。

```html
<label for="nickname">模拟昵称（可留空）</label>
<input id="nickname" name="nickname" type="text"
       autocomplete="nickname">
<!-- autocomplete 不会自动建立必填或格式约束；留空仍可回显 nickname=""。 -->
```

配套文件：[scripts/11-form-submission-and-validation/pattern.html](scripts/11-form-submission-and-validation/pattern.html) · [浏览器预览](http://127.0.0.1:8011/pattern.html)

## 7 客户端与服务端的校验边界

原生校验帮助使用者改错，不能让服务端直接信任输入。HTML 可被修改，请求也可以绕过页面发送。

- &lt;form&gt; 的 novalidate：跳过正常提交的交互式校验。
- 提交按钮的 formnovalidate：只跳过本次交互式校验。

约束页提供两个按钮用于对照；先尝试正常提交，再让别名留空并使用“跳过校验”按钮。

```html
<button type="submit" name="intent" value="checked">按原生约束提交</button>
<button type="submit" name="intent" value="unchecked"
        formnovalidate>跳过校验并回显（仅作对照）</button>
<!-- 别名留空时点“跳过校验”应仍可发出请求；回显不代表这些数据业务上有效。 -->
```

配套文件：[scripts/11-form-submission-and-validation/constraints.html](scripts/11-form-submission-and-validation/constraints.html) · [浏览器预览](http://127.0.0.1:8011/constraints.html)

真实服务必须独立检查字段、类型、范围、长度、格式及业务条件；只读和隐藏字段同样不能直接信任。文件选择提示、客户端校验和 HTTPS 各有用途，互不替代。

本章服务仅限制请求体大小并回显，返回 200 只说明请求得到处理，不表示业务数据有效。另一浏览器的提示、移动端键盘和自动填充需要在对应环境中另行检查。

## 本章小结

- action 决定目的地，method 影响数据位置，enctype 决定 POST 编码；提交按钮可覆盖本次设置。
- 实际条目取决于名称、状态和提交者。同名条目可重复，空值与没有字段不同。
- 文件内容需要 multipart；原生约束有适用类型与输入条件，autocomplete 和 inputmode 只是输入提示。
- 能提交、收到 200 和业务校验通过是三件事，客户端校验不能代替服务端校验。

## 练习

在 scripts/11-form-submission-and-validation/ 内复制对应页面后修改。服务允许下面三个副本名称，浏览器分别使用 [practice-submission.html](http://127.0.0.1:8011/practice-submission.html)、[practice-constraints.html](http://127.0.0.1:8011/practice-constraints.html)、[practice-pattern.html](http://127.0.0.1:8011/practice-pattern.html)；复制前这些地址返回 404。

（1）复制 submission.html 为 practice-submission.html，给“有 id、没有 name”的字段增加 name="local_note"，取消全部 topic 勾选。检查：GET 与 POST 都有 local_note、没有 topic，每次只有本次按钮的 intent；memo 留空时仍有空值条目。

（2）复制 constraints.html 为 practice-constraints.html，把份数改成 0.5～2.5、间隔 0.5。检查：其他字段合法时，0.5、1.5、2.5 可提交，0.7 不可；备注留空可提交，手动输入 ab 不可。

（3）复制 pattern.html 为 practice-pattern.html，让大写编号允许下划线。检查：AB_12、AB-12 通过，ab_12、AB/12 不通过；六位数字编号仍保留 001234。

（4）使用 upload.html 上传 sample-note.txt。检查：请求体包含字段名、文件名、boundary 和两行文件内容；指出总请求体大小与文件大小的区别。

### 提示

第一题先预测条目，再用 Network 与回显页交叉核对。第二题同步修改 min、max、step、value，保持其他输入合法。第三题保留字面连字符的转义。练习副本用后可删除，原页面、素材和服务器源码保留。

## 参考与引用来源

- WHATWG HTML：[提交属性](https://html.spec.whatwg.org/multipage/form-control-infrastructure.html#form-submission-attributes)、[隐式提交](https://html.spec.whatwg.org/multipage/form-control-infrastructure.html#implicit-submission)、[条目构造](https://html.spec.whatwg.org/multipage/form-control-infrastructure.html#constructing-the-form-data-set)、[名称和值转换](https://html.spec.whatwg.org/multipage/form-control-infrastructure.html#convert-to-a-list-of-name-value-pairs)、[multipart](https://html.spec.whatwg.org/multipage/form-control-infrastructure.html#multipart-form-data)：提交地址与方法、按钮覆盖、字段取舍、重名、空值和文件编码；[input 属性](https://html.spec.whatwg.org/multipage/input.html#common-input-element-attributes)的 required、min/max、step、长度与 [pattern](https://html.spec.whatwg.org/multipage/input.html#the-pattern-attribute)；[表单客户端校验](https://html.spec.whatwg.org/multipage/forms.html#client-side-form-validation)的服务端责任。
- WHATWG URL：[application/x-www-form-urlencoded](https://url.spec.whatwg.org/#application/x-www-form-urlencoded)：UTF-8、百分号编码、空格与加号。
- MDN：[Sending form data](https://developer.mozilla.org/en-US/docs/Learn_web_development/Extensions/Forms/Sending_and_retrieving_form_data#the_method_attribute)的 GET/POST 与传输边界；[button 提交属性](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Elements/button#attributes)、[file](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Elements/input/file#limiting_accepted_file_types)、[readonly](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Attributes/readonly#attribute_interactions)、[disabled](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Attributes/disabled#overview)；[Constraint validation process](https://developer.mozilla.org/en-US/docs/Web/HTML/Guides/Constraint_validation#constraint_validation_process)的适用约束、UTF-16 长度、用户输入与脚本赋值、跳过校验；[step](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Attributes/step#syntax)、[pattern](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Attributes/pattern#overview)、[autocomplete](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Attributes/autocomplete#description)、[inputmode](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Global_attributes/inputmode#value)的取值、边界与支持条件。
- W3C WAI：[Validating Input](https://www.w3.org/WAI/tutorials/forms/validation/)：可见要求、必填标识与浏览器提示差异。
- Chrome for Developers：[Network features reference](https://developer.chrome.com/docs/devtools/network/reference/#payload)：Preserve log、请求标头、载荷与原始编码查看。
- Python 3.12：[http.server](https://docs.python.org/3.12/library/http.server.html#http.server.BaseHTTPRequestHandler)：本机回显服务使用的请求处理器、请求流与响应方法。